In [0]:
# ADLS Gen2 paths

storage_account = "stedukronretaildl"

bronze_base = (
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/processed"
)

silver_base = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/processed"
)

print("Bronze path:", bronze_base)
print("Silver path:", silver_base)

In [0]:
manifest_path = (
    "abfss://bronze@stedukronretaildl.dfs.core.windows.net/raw/"
    "blob/../manifest.json"
)

In [0]:
for source in ["blob", "api", "sql", "adls"]:
    print(f"\n========== {source.upper()} ==========")

    source_path = f"{bronze_base}/{source}"

    for item in dbutils.fs.ls(source_path):
        df = spark.read.format("delta").load(item.path)

        print(f"\nTable: {item.name}")
        print(df.dtypes)

In [0]:

from pyspark.sql.functions import (
    col, trim, to_date, to_timestamp
)

# Primary keys for the 32 datasets
primary_keys = {
    "blob": {
        "brands": ["brand_id"],
        "categories": ["category_id"],
        "channels": ["channel_id"],
        "product_attributes": ["attribute_id"],
        "product_prices": ["price_id"],
        "products": ["product_id"],
        "promotions": ["promotion_id"],
        "stores": ["store_id"]
    },
    "api": {
        "campaigns": ["campaign_id"],
        "cart_items": ["cart_item_id"],
        "carts": ["cart_id"],
        "clickstream_events": ["event_id"],
        "delivery_events": ["delivery_event_id"],
        "product_reviews": ["review_id"],
        "support_tickets": ["ticket_id"],
        "web_sessions": ["session_id"]
    },
    "sql": {
        "customer_addresses": ["address_id"],
        "customers": ["customer_id"],
        "employees": ["employee_id"],
        "order_items": ["order_item_id"],
        "orders": ["order_id"],
        "payments": ["payment_id"],
        "return_items": ["return_item_id"],
        "returns": ["return_id"]
    },
    "adls": {
        "inventory_snapshots": ["snapshot_id"],
        "purchase_order_items": ["purchase_order_item_id"],
        "purchase_orders": ["purchase_order_id"],
        "shipment_items": ["shipment_item_id"],
        "shipments": ["shipment_id"],
        "stock_movements": ["movement_id"],
        "suppliers": ["supplier_id"],
        "warehouses": ["warehouse_id"]
    }
}

# Columns that should be converted to DATE
date_columns = [
    "start_date", "end_date", "review_date",
    "order_date", "payment_date", "return_date",
    "snapshot_date", "shipment_date", "movement_date"
]

# Columns that should be converted to TIMESTAMP
timestamp_columns = [
    "created_at", "occurred_at",
    "event_time", "started_at"
]


def transform_to_silver(df, source, table_name):

    # 1. Trim whitespace from string columns
    for column_name, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(
                column_name,
                trim(col(column_name))
            )

    # 2. Convert date columns
    for column_name in date_columns:
        if column_name in df.columns:
            df = df.withColumn(
                column_name,
                to_date(col(column_name))
            )

    # 3. Convert timestamp columns
    for column_name in timestamp_columns:
        if column_name in df.columns:
            df = df.withColumn(
                column_name,
                to_timestamp(col(column_name))
            )

    # 4. Remove duplicate records using primary keys
    keys = primary_keys[source][table_name]

    before_count = df.count()

    df = df.dropDuplicates(keys)

    after_count = df.count()

    # 5. Write the cleaned data to Silver
    output_path = f"{silver_base}/{source}/{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .save(output_path)
    )

    print(
        f"{source}/{table_name}: "
        f"Before={before_count}, "
        f"After={after_count}, "
        f"Duplicates removed={before_count - after_count}"
    )


print("Silver transformation function created!")


In [0]:

for source in ["blob", "api", "sql", "adls"]:

    source_path = f"{bronze_base}/{source}"

    for item in dbutils.fs.ls(source_path):

        table_name = item.name.rstrip("/")

        print(f"Processing {source}/{table_name}")

        df = spark.read.format("delta").load(item.path)

        transform_to_silver(
            df,
            source,
            table_name
        )

print("Silver transformations completed!")


In [0]:
print(spark.conf.get(
    "fs.azure.account.key.stedukronretaildl.dfs.core.windows.net",
    "NOT_CONFIGURED"
))

In [0]:
silver_test_path = (
    "abfss://silver@stedukronretaildl.dfs.core.windows.net/test"
)

dbutils.fs.mkdirs(silver_test_path)

print("Silver storage write test successful!")

In [0]:
for source in ["blob", "api", "sql", "adls"]:

    path = f"{silver_base}/{source}"

    datasets = dbutils.fs.ls(path)

    print(f"{source.upper()}: {len(datasets)} datasets")

    for item in datasets:
        df = spark.read.format("delta").load(item.path)
        print(item.name, df.count())